In [1]:
# 1. 라이브러리 불러오기 및 실습 데이터 생성
# 실습을 위해 '상품 카테고리'와 '등급'이라는 두 가지 범주형 변수를 가진 가상 데이터를 생성.

import pandas as pd
import numpy as np

# 실습용 샘플 데이터 생성
data = {
    '상품ID': ['P01', 'P02', 'P03', 'P04', 'P05'],
    '상품카테고리': ['TV', '냉장고', '컴퓨터', 'TV', '컴퓨터'],
    '고객등급': ['Silver', 'Gold', 'Bronze', 'Silver', 'Gold']
}

df = pd.DataFrame(data)
print("--- 원본 데이터프레임 ---")
print(df)

--- 원본 데이터프레임 ---
  상품ID 상품카테고리    고객등급
0  P01     TV  Silver
1  P02    냉장고    Gold
2  P03    컴퓨터  Bronze
3  P04     TV  Silver
4  P05    컴퓨터    Gold


In [2]:
# 2. 라벨 인코딩 (Label Encoding)
# •	원리: 각 문자열 값을 고유한 정수(0, 1, 2...)로 일대일 매핑하여 가로 길이를 유지하는 방식.
# •	주의점: 컴퓨터는 숫자의 크기(0 < 1 < 2)를 보고 무의미한 서열/가중치 관계가 있다고 오해할 수 있다. 따라서 순서에 의미가 없는 변수(예: TV, 냉장고)에는 사용하면 위험하며, 순서가 중요한 데이터에 주로 적합.

from sklearn.preprocessing import LabelEncoder

df_label = df.copy()

# 1. 인코더 객체 생성
le = LabelEncoder()

# 2. 데이터 학습 및 변환 (fit_transform)
# 여기서는 '고객등급' 변수에 적용해 봅니다. (Bronze=0, Gold=1, Silver=2 순서로 인덱싱됨)
df_label['고객등급_encoded'] = le.fit_transform(df_label['고객등급'])

print("--- 1. Label Encoding 적용 결과 ---")
print(df_label[['상품ID', '고객등급', '고객등급_encoded']])

# 팁: 어떤 문자가 어떤 숫자로 바뀌었는지 확인하기
print("\n인코딩 매핑 클래스 확인:", le.classes_)

--- 1. Label Encoding 적용 결과 ---
  상품ID    고객등급  고객등급_encoded
0  P01  Silver             2
1  P02    Gold             1
2  P03  Bronze             0
3  P04  Silver             2
4  P05    Gold             1

인코딩 매핑 클래스 확인: ['Bronze' 'Gold' 'Silver']


In [3]:
# 3. 원핫 인코딩 (One-Hot Encoding)
# •	원리: 고유한 값의 개수만큼 새로운 열(Column)을 추가한 뒤, 해당되는 위치에만 1을 주고 나머지는 0으로 채우는 방식.
# •	특징: 고유값 사이에 서열이나 거리감이 생기지 않으므로 트리 계열이 아닌 선형 모델이나 신경망에서 필수적으로 사용. 다만 고유값 종류가 너무 많으면 열이 폭발적으로 늘어나는 '차원의 저주'가 생길 수 있다.

# 방법 A: Pandas의 get_dummies() 활용 (실무 선호, 직관적)
# 데이터프레임의 문자열 컬럼을 한 번에 원핫 인코딩하여 결과 뷰까지 다듬어주기 때문에 가장 널리 쓰입니다.

df_oh_pandas = df.copy()

# columns 옵션에 지정한 열만 원핫 인코딩으로 확장됩니다.
df_encoded_pd = pd.get_dummies(df_oh_pandas, columns=['상품카테고리'], dtype=int)

print("--- 2-A. Pandas get_dummies() 적용 결과 ---")
print(df_encoded_pd)

--- 2-A. Pandas get_dummies() 적용 결과 ---
  상품ID    고객등급  상품카테고리_TV  상품카테고리_냉장고  상품카테고리_컴퓨터
0  P01  Silver          1           0           0
1  P02    Gold          0           1           0
2  P03  Bronze          0           0           1
3  P04  Silver          1           0           0
4  P05    Gold          0           0           1


In [4]:
# 방법 B: Scikit-learn의 OneHotEncoder 활용 (머신러닝 파이프라인용)
# 추후 새로운 검증 데이터(Test Data)가 들어왔을 때 동일한 규칙(학습된 열 구조)을 유지해야 하는 파이프라인 구축 시 필수적.

from sklearn.preprocessing import OneHotEncoder

df_oh_sklearn = df.copy()

# 1. 인코더 객체 생성 (sparse_output=False로 지정해야 넘파이 행렬로 직관적으로 반환됨)
oh_encoder = OneHotEncoder(sparse_output=False)

# 2. 2차원 배열 형태로 입력하여 학습 및 변환
oh_result = oh_encoder.fit_transform(df_oh_sklearn[['상품카테고리']])

# 3. 변환된 데이터를 데이터프레임으로 결합
encoded_cols = oh_encoder.get_feature_names_out(['상품카테고리'])
df_oh_res = pd.DataFrame(oh_result, columns=encoded_cols, dtype=int)

# 원본 ID와 결합하여 출력
print("\n--- 2-B. Scikit-learn OneHotEncoder 적용 결과 ---")
print(pd.concat([df_oh_sklearn['상품ID'], df_oh_res], axis=1))


--- 2-B. Scikit-learn OneHotEncoder 적용 결과 ---
  상품ID  상품카테고리_TV  상품카테고리_냉장고  상품카테고리_컴퓨터
0  P01          1           0           0
1  P02          0           1           0
2  P03          0           0           1
3  P04          1           0           0
4  P05          0           0           1
